In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

## Install YOLO11 via Ultralytics

In [ ]:
import ultralytics
from ultralytics import settings

settings.reset()

ultralytics.checks()

In [ ]:
ultralytics.settings['tensorboard'] = True

## Fine-tune YOLO11 on custom dataset

In [ ]:
!mkdir {HOME}\datasets
%cd {HOME}\datasets

from roboflow import Roboflow
rf = Roboflow(api_key="")
project = rf.workspace("").project("")
version = project.version(1)
dataset = version.download("yolov11")                

print(dataset.location)

%cd {HOME}

## Custom Training

```py
tensorboard --logdir runs/segment/train
```

In [ ]:
from ultralytics import YOLO

# Load the last saved model
model = YOLO("yolo11s-seg.pt")  # or use your custom path

# Resume training from where it left off
model.train(
    data=f"{dataset.location}/data.yaml",
    imgsz=640,
    epochs=500,
    batch=16,
    plots=True,
    save=True,
    save_period=10,
    patience=30,
    cache="disk",
    # lr0=1e-5,
    seed=42
)


**NOTE:** The results of the completed training are saved in `{HOME}/runs/detect/train/`. Let's examine them.

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/segment/train/confusion_matrix.png')

## Validate fine-tuned model

BEST PT

In [ ]:
import pandas as pd

# Load the training log
results = pd.read_csv('runs/segment/train/results.csv')

# Strip spaces
results.columns = results.columns.str.strip()

# Calculate fitness
results["fitness"] = results["metrics/mAP50(B)"] * 0.1 + results["metrics/mAP50-95(B)"] * 0.9

# Find the epoch with the highest fitness
best_epoch = results['fitness'].idxmax() + 1

print(f"Best model was saved at epoch: {best_epoch}")

## Inference with custom model

In [ ]:
from ultralytics import YOLO
model = YOLO("runs/segment/train/weights/best.pt")

directory = f"{dataset.location}/valid/images"
pred_val_dir = "val"

model.predict(source=directory, save=True, project=pred_val_dir, verbose=False, iou=0.1, conf=0.1)